# R matrix reconstruction from beam-size data

This notebook follows the equation from the multiple-wire / multi-screen emittance measurement method.
The same equation can be used for a quadrupole scan.

For one transverse plane, the beam matrix at the starting point `s0` is

$$
\Sigma_0 =
\epsilon
\begin{pmatrix}
\beta_0 & -\alpha_0 \\
-\alpha_0 & \gamma_0
\end{pmatrix},
\qquad
\gamma_0 = \frac{1 + \alpha_0^2}{\beta_0}
$$

The linear transport from `s0` to screen `i` is

$$
R^{(i)} =
\begin{pmatrix}
R_{11}^{(i)} & R_{12}^{(i)} \\
R_{21}^{(i)} & R_{22}^{(i)}
\end{pmatrix}.
$$

The beam matrix at screen `i` is transported as

$$
\Sigma_i = R^{(i)} \Sigma_0 \left(R^{(i)}\right)^T.
$$

The measured beam size squared is the first element of this matrix:

$$
\sigma_i^2 = \Sigma_{i,11}.
$$

After expanding this, we get

$$
\sigma_i^2 =
\left(R_{11}^{(i)}\right)^2 \beta_0 \epsilon
+ 2R_{11}^{(i)}R_{12}^{(i)}\left(-\alpha_0\epsilon\right)
+ \left(R_{12}^{(i)}\right)^2 \gamma_0\epsilon.
$$

So for many screens or many measurements, this becomes a linear system:

$$
\begin{pmatrix}
\sigma_1^2 \\
\sigma_2^2 \\
\sigma_3^2 \\
\vdots \\
\sigma_n^2
\end{pmatrix}
=
\begin{pmatrix}
\left(R_{11}^{(1)}\right)^2 & 2R_{11}^{(1)}R_{12}^{(1)} & \left(R_{12}^{(1)}\right)^2 \\
\left(R_{11}^{(2)}\right)^2 & 2R_{11}^{(2)}R_{12}^{(2)} & \left(R_{12}^{(2)}\right)^2 \\
\left(R_{11}^{(3)}\right)^2 & 2R_{11}^{(3)}R_{12}^{(3)} & \left(R_{12}^{(3)}\right)^2 \\
\vdots & \vdots & \vdots \\
\left(R_{11}^{(n)}\right)^2 & 2R_{11}^{(n)}R_{12}^{(n)} & \left(R_{12}^{(n)}\right)^2
\end{pmatrix}
\begin{pmatrix}
\beta_0\epsilon \\
-\alpha_0\epsilon \\
\gamma_0\epsilon
\end{pmatrix}.
$$

In our dataset, we do the inverse problem: we know many input beam parameters and many output beam sizes, so we fit the coefficients multiplying

$$
\beta\epsilon, \qquad -\alpha\epsilon, \qquad \gamma\epsilon.
$$

Therefore, for each screen we fit

$$
\sigma_i^2 = A_i\beta\epsilon + B_i(-\alpha\epsilon) + C_i\gamma\epsilon + D_i.
$$

For ideal uncoupled linear transport:

$$
A_i = \left(R_{11}^{(i)}\right)^2,
\qquad
B_i = 2R_{11}^{(i)}R_{12}^{(i)},
\qquad
C_i = \left(R_{12}^{(i)}\right)^2.
$$

The extra term `D_i` is only a numerical / offset term. Ideally it should be close to zero.

For the horizontal plane we solve

$$
B_x = C_x R_x^T,
$$

where

$$
C_x =
\begin{pmatrix}
\beta_x\epsilon_x & -\alpha_x\epsilon_x & \gamma_x\epsilon_x & 1
\end{pmatrix}
$$

and

$$
B_x =
\begin{pmatrix}
\sigma_{x,OTR0}^2 & \sigma_{x,OTR1}^2 & \sigma_{x,OTR2}^2 & \sigma_{x,OTR3}^2
\end{pmatrix}.
$$

Similarly for the vertical plane:

$$
B_y = C_y R_y^T.
$$

Important: from beam-size data alone we recover the combinations

$$
R_{11}^2, \qquad 2R_{11}R_{12}, \qquad R_{12}^2,
$$

not a fully signed unique transfer matrix. The global sign convention of `R11` and `R12` cannot be fixed from beam sizes only.


In [ ]:
import numpy as np
import RF_Track as rft

I = np.loadtxt("input_2000.txt")
O = np.loadtxt("output_2000.txt")

mass = rft.electronmass  # MeV/c^2
momentum = 1300  # MeV/c

beta_gamma = momentum / mass

emitt_x = I[:, 0] / beta_gamma  # mm.mrad, geometric emittance
emitt_y = I[:, 3] / beta_gamma  # mm.mrad, geometric emittance
beta_x = I[:, 1]
beta_y = I[:, 4]
alpha_x = I[:, 2]
alpha_y = I[:, 5]

S1_x = O[:, 0]  # mm
S1_y = O[:, 1]
S2_x = O[:, 2]
S2_y = O[:, 3]
S3_x = O[:, 4]
S3_y = O[:, 5]
S4_x = O[:, 6]
S4_y = O[:, 7]

gamma_x = (1 + alpha_x**2) / beta_x
gamma_y = (1 + alpha_y**2) / beta_y

ones = np.ones(len(I))

Cx = np.column_stack((beta_x * emitt_x, -alpha_x * emitt_x, gamma_x * emitt_x, ones))
Cy = np.column_stack((beta_y * emitt_y, -alpha_y * emitt_y, gamma_y * emitt_y, ones))

Bx = np.column_stack((S1_x, S2_x, S3_x, S4_x)) ** 2
By = np.column_stack((S1_y, S2_y, S3_y, S4_y)) ** 2

Rx = Bx.T @ np.linalg.pinv(Cx.T)
Ry = By.T @ np.linalg.pinv(Cy.T)

print("Rx =")
print(Rx)

print("Ry =")
print(Ry)